# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated model output into a human-reviewed content action playbook.
It exports the ranked queue and figures to `work/outputs/` for the deployed research paper.

> Built on the [FlyRank ML Internship](https://flyrank.ai) dataset. All data is anonymized.

## 1. Ranked actions + reason codes

The model produces a probability score for each page. I combine that with rule-based signals
(impressions, CTR, position, engagement, content age) to produce a **final refresh score**
(0-100), a **suggested action**, and human-readable **reason codes**.

**Action taxonomy:**

| Action | Meaning | When to use |
|---|---|---|
| `refresh` | Review and update content | Model flags decline risk with demand |
| `refresh_and_review_ctr` | Refresh + check snippet/metadata | Declining with good visibility but low CTR |
| `refresh_and_review_engagement` | Refresh + check content quality | Traffic exists but engagement is low |
| `expand_and_refresh` | Expand thin content then refresh | Visible page with thin word count |
| `monitor` | No action now, track next cycle | Model scored low |
| `protect` | Guard a growing asset | Growing page with strong visibility |

**Reason codes:**

| Code | Trigger |
|---|---|
| `model_decline_risk` | Model probability >= 0.65 |
| `visible_model_opportunity` | Model probability >= 0.50 + impressions >= 500 |
| `declining_with_demand` | trend_direction=down + impressions >= 100 |
| `stale_visible_page` | days_since_last_update >= 180 + impressions >= 500 |
| `thin_visible_page` | word_count < 1200 + impressions >= 250 |
| `low_ctr_visible_page` | impressions >= 500, position <= 20, CTR < 0.5% |
| `low_engagement_visible_page` | sessions >= 30, engagement_rate < 30% or scroll_rate < 30% |
| `page_one_decay_risk` | position <= 10, age >= 180 days |
| `growing_asset_protect` | trend_direction=up + impressions >= 500 |
| `general_refresh_review` | Fallback |

**Archetype to action mapping:**

| Archetype | Typical signals | Suggested action |
|---|---|---|
| Declining leader | High impressions, declining trend, old | refresh or refresh_and_review_ctr |
| Underperforming contender | Good position, low CTR, some impressions | refresh_and_review_ctr |
| Thin but visible | Low word count, moderate impressions | expand_and_refresh |
| Engagement gap | Traffic but low scroll/engagement | refresh_and_review_engagement |
| Stable performer | No decline signals, moderate visibility | monitor |
| Growing asset | Upward trend, good visibility | protect |

**Decay / refresh insight:** The model's top features are `days_with_impressions` (38.4%),
`content_age_days` (19.8%), and `avg_position` (11.8%). Age alone is a weak signal (the
stale-first baseline lost to random), but age combined with sustained visibility and declining
position is a strong decline pattern. The refresh opportunity is clearest where a page has
history (many days with impressions) and is now slipping in position.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path('..') if (Path('..') / 'data').exists() else Path('.')
if not (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    ROOT = Path('/Users/keremozcan/.openclaw/workspace/flyrank-ml-internship-submission')

FEATURE_PATH = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
BASELINE_PATH = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'
WORK_OUTPUTS = ROOT / 'work' / 'outputs'
WORK_FIGURES = ROOT / 'work' / 'figures'
RANDOM_STATE = 42

WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)
WORK_FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FEATURE_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)
print(f'Rows: {len(df):,}  Clients: {df["client_id"].nunique()}  Base rate: {df["is_declining_label"].mean():.3f}')

In [ ]:
# --- Build feature matrix and train gradient boosting ---

MODEL_NUMERIC = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
MODEL_CATEGORICAL = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

def build_feature_matrix(frame):
    num = frame[MODEL_NUMERIC].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[MODEL_CATEGORICAL].fillna('unknown').astype(str)
    encoded = pd.get_dummies(cat, prefix=MODEL_CATEGORICAL, dummy_na=False, dtype=float)
    return pd.concat([num.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)

X = build_feature_matrix(df)
y = df['is_declining_label'].astype(int)
feature_cols = list(X.columns)

model = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
model.fit(X, y)
probabilities = model.predict_proba(X)[:, 1]

print(f'Model trained on {len(df):,} rows')
print(f'Probability range: {probabilities.min():.3f} - {probabilities.max():.3f}')

In [ ]:
# --- Build the action queue with reason codes ---

def normalize_series(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-10)

queue = df.copy()
queue['model_probability'] = probabilities

baseline_lookup = baseline_df.set_index('content_id')['baseline_refresh_score']
queue['baseline_score'] = queue['content_id'].map(baseline_lookup).fillna(0)
queue['baseline_score_norm'] = normalize_series(queue['baseline_score'])
queue['final_refresh_score'] = (100 * (0.70 * queue['model_probability'] + 0.30 * queue['baseline_score_norm'])).clip(0, 100)

def build_reason_codes(row):
    reasons = []
    prob = row['model_probability']
    imp = row['impressions_90d']
    pos = row['avg_position']
    ctr = row['ctr']
    age = row['content_age_days']
    update = row['days_since_last_update']
    wc = row['word_count']
    sessions = row['sessions_90d']
    eng = row['engagement_rate']
    scroll = row['scroll_rate']
    trend = str(row['trend_direction']).lower()
    
    if prob >= 0.65: reasons.append('model_decline_risk')
    if prob >= 0.50 and imp >= 500: reasons.append('visible_model_opportunity')
    if trend == 'down' and imp >= 100: reasons.append('declining_with_demand')
    if update >= 180 and imp >= 500: reasons.append('stale_visible_page')
    if wc > 0 and wc < 1200 and imp >= 250: reasons.append('thin_visible_page')
    if imp >= 500 and 0 < pos <= 20 and ctr < 0.5: reasons.append('low_ctr_visible_page')
    if sessions >= 30 and ((eng > 0 and eng < 30) or (scroll > 0 and scroll < 30)): reasons.append('low_engagement_visible_page')
    if 0 < pos <= 10 and age >= 180: reasons.append('page_one_decay_risk')
    if trend == 'up' and imp >= 500: reasons.append('growing_asset_protect')
    return '|'.join(reasons) if reasons else 'general_refresh_review'

queue['reason_codes'] = queue.apply(build_reason_codes, axis=1)

def suggest_action(row):
    reasons = set(str(row['reason_codes']).split('|'))
    if 'growing_asset_protect' in reasons and 'model_decline_risk' not in reasons:
        return 'protect'
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons and ('model_decline_risk' in reasons or 'declining_with_demand' in reasons):
        return 'refresh_and_review_ctr'
    if 'low_engagement_visible_page' in reasons and ('model_decline_risk' in reasons or 'declining_with_demand' in reasons):
        return 'refresh_and_review_engagement'
    if {'model_decline_risk', 'declining_with_demand', 'stale_visible_page', 'visible_model_opportunity'}.intersection(reasons):
        return 'refresh'
    return 'monitor'

queue['suggested_action'] = queue.apply(suggest_action, axis=1)

high_threshold = queue['final_refresh_score'].quantile(0.8)
medium_threshold = queue['final_refresh_score'].quantile(0.5)

def confidence_label(row):
    if row['final_refresh_score'] >= high_threshold and row['impressions_90d'] >= 500 and row['model_probability'] >= 0.5:
        return 'high'
    if row['final_refresh_score'] >= medium_threshold:
        return 'medium'
    return 'low'

queue['confidence'] = queue.apply(confidence_label, axis=1)
queue = queue.sort_values(['final_refresh_score', 'impressions_90d', 'sessions_90d'], ascending=[False, False, False]).reset_index(drop=True)
queue['rank'] = queue.index + 1

# Summary
print('ACTION QUEUE SUMMARY')
print('=' * 60)
print(f'Total pages: {len(queue):,}')
print(f'Score range: {queue["final_refresh_score"].min():.1f} - {queue["final_refresh_score"].max():.1f}')
print(f'High threshold: {high_threshold:.1f}  Medium threshold: {medium_threshold:.1f}')
print()
print('Action distribution:')
for action, count in queue['suggested_action'].value_counts().items():
    print(f'  {action:<35} {count:>6,}  ({count/len(queue)*100:.1f}%)')
print()
print('Confidence:')
for conf, count in queue['confidence'].value_counts().items():
    print(f'  {conf:<10} {count:>6,}  ({count/len(queue)*100:.1f}%)')
print()
all_reasons = []
for rc in queue['reason_codes']:
    all_reasons.extend(str(rc).split('|'))
reason_counts = pd.Series(all_reasons).value_counts().head(10)
print('Top 10 reason codes:')
for reason, count in reason_counts.items():
    print(f'  {reason:<35} {count:>6,}')

In [ ]:
# --- Top 20 ranked pages ---
view_cols = ['rank', 'final_refresh_score', 'model_probability', 'suggested_action', 'confidence',
             'reason_codes', 'is_declining_label', 'impressions_90d', 'ctr', 'avg_position',
             'content_age_days', 'days_since_last_update', 'word_count', 'content_type', 'trend_direction']

print('TOP 20 RANKED PAGES:')
print('=' * 110)
for _, row in queue[view_cols].head(20).iterrows():
    print(f'#{row["rank"]:>3}  score={row["final_refresh_score"]:>5.1f}  prob={row["model_probability"]:.3f}  '
          f'action={row["suggested_action"]:<30}  conf={row["confidence"]}')
    print(f'      reasons: {row["reason_codes"]}')
    print(f'      trend={row["trend_direction"]}  imp={row["impressions_90d"]:,}  ctr={row["ctr"]:.2f}%  '
          f'pos={row["avg_position"]}  age={row["content_age_days"]}d  words={row["word_count"]}  type={row["content_type"]}')
    print()

## 2. Intended use and limits

**Who uses this:** A content editor or SEO manager with limited review capacity (e.g., 50 pages
per cycle). They open the ranked queue, start from the top, and work down.

**What they do with it:**
1. Open `work/outputs/refresh_queue.csv`
2. Take the top N rows (N = review capacity per cycle)
3. For each row: inspect the page, read the reason codes, check whether the suggested action
   makes sense in editorial context, then decide: update, expand, revise metadata, or leave alone
4. Track which actions improved traffic over the next 30 days

**Where it stops being valid:**

- **Not a prospective predictor.** The 90-day activity metrics overlap with the label window.
  The model uses current-window signals to rank current-window decline. It cannot tell you a
  page *will* decline — only that it currently shows decline-associated signals.
- **Not a causal model.** No experiment was run. The model ranks review priority; it does not
  measure refresh outcomes. A page may be declining for reasons a refresh cannot fix.
- **Not a production system.** This is a research artifact on a 30K-page teaching dataset.
- **No client-specific calibration.** The model was trained across all clients.
- **Base rate is 54.2%.** More than half the pages are labeled declining. The model's job is
  prioritization, not detection.

**Cost/value thinking:**

| Action | Editor cost | Expected value | When it pays off |
|---|---|---|---|
| `refresh_and_review_ctr` | Low (metadata tweak) | Medium-high | Page has traffic but low click capture |
| `refresh` | Medium (content update) | Medium | Page is declining with demand |
| `refresh_and_review_engagement` | Medium-high (content + UX) | Medium | Traffic exists but visitors bounce |
| `expand_and_refresh` | High (significant rewrite) | Low-medium | Thin page with visibility — only if topic needs depth |
| `protect` | Low (monitoring only) | High | Growing page — don't touch, just track |
| `monitor` | Zero (no action) | Zero | No signals — wait for next cycle |

## 3. Human review + the no-go list

**What a person must check before acting:**

1. **Read the page.** The model has never seen the content — it only sees numeric signals.
   A page flagged as "thin" might be intentionally concise.

2. **Check the query intent.** Low CTR might be structural (featured snippets answer directly).

3. **Check for cannibalization.** Two pages targeting the same query will split impressions.

4. **Check seasonality.** A page declining in December might be seasonal.

5. **Check for recent updates.** The dataset is a snapshot; freshness data may lag.

**The no-go list — what should NEVER be automated:**

| No-go | Why |
|---|---|
| Auto-publishing refreshes | The model ranks priority; it does not write content |
| Auto-deleting pages | `monitor` means no action now, not delete |
| Auto-redirecting or merging | Requires query-level analysis the model cannot do |
| Using model probability as a KPI | Track refresh outcomes, not model scores |
| Applying to clients not in training data | Retrain first |
| Acting on pages with < 100 impressions | False negative cluster — not worth editor time |
| Ignoring the trend_direction | If model flags a page with trend=up, that's a false positive |

## 4. Monitoring / retrain triggers

**What would tell you the recommendations went stale?**

| Signal | Threshold | Action |
|---|---|---|
| Precision@50 drift | Drops below 0.60 (from 0.90) | Retrain on fresh data |
| Label distribution shift | Base rate changes by > 10pp | Investigate data pipeline or portfolio shift |
| Feature importance shift | Top feature changes | Retrain — signal structure changed |
| False positive rate spike | FP rate on top-50 exceeds 40% | Review queue manually; retrain |
| New content types | New content_type in production | Add to features; retrain |
| Seasonal cycles | Quarterly review | Compare recommendations against editor feedback |

**Retrain cadence:**
- Minimum: Quarterly (every 90 days)
- Recommended: Monthly with full warehouse refresh
- Trigger-based: Any signal above should trigger immediate retrain

**Monitoring metrics:**
1. Model performance: Precision@50, ROC AUC on held-out client set
2. Editor feedback: % of top-50 recommendations acted on vs dismissed
3. Outcome tracking: Did refreshed pages improve 30 days post-refresh?
4. Queue health: Is the action distribution stable?

## 5. Exports for the paper

The notebook exports the ranked queue and summary metrics to `work/outputs/`. Figures go to
`work/figures/`. The queue CSV is gitignored (CI leak-guard); the notebook regenerates it.
Figures and metrics JSONs are committed as receipts.

In [ ]:
# --- Export: ranked queue to work/outputs/ ---
export_cols = [
    'rank', 'content_id', 'client_id', 'final_refresh_score', 'model_probability',
    'confidence', 'suggested_action', 'reason_codes', 'is_declining_label',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
    'content_age_days', 'days_since_last_update', 'word_count',
    'trend_direction', 'content_type', 'main_intent', 'impression_tier', 'position_tier',
]

export_queue = queue[export_cols].copy()
export_queue.to_csv(WORK_OUTPUTS / 'refresh_queue.csv', index=False)
export_queue.head(50).to_csv(WORK_OUTPUTS / 'refresh_queue_top50.csv', index=False)
export_queue.head(100).to_csv(WORK_OUTPUTS / 'refresh_queue_top100.csv', index=False)
print(f'Exported queue: {len(export_queue):,} rows')
print(f'Exported top-50 and top-100')

In [ ]:
# --- Export: action summary JSON ---
action_summary = {
    'total_pages': int(len(queue)),
    'model': 'gradient_boosting',
    'score_formula': '0.70 * model_probability + 0.30 * normalized_baseline',
    'score_range': [float(queue['final_refresh_score'].min()), float(queue['final_refresh_score'].max())],
    'confidence_thresholds': {'high': float(high_threshold), 'medium': float(medium_threshold)},
    'action_distribution': {k: int(v) for k, v in queue['suggested_action'].value_counts().items()},
    'confidence_distribution': {k: int(v) for k, v in queue['confidence'].value_counts().items()},
    'top_reason_codes': {k: int(v) for k, v in reason_counts.head(10).items()},
    'top_50_declining_rate': float(export_queue.head(50)['is_declining_label'].mean()),
    'top_100_declining_rate': float(export_queue.head(100)['is_declining_label'].mean()),
    'base_rate': float(df['is_declining_label'].mean()),
    'actions': {
        'refresh': 'Review and update content — model flags decline risk with demand',
        'refresh_and_review_ctr': 'Refresh + check snippet/metadata — low CTR with visibility',
        'refresh_and_review_engagement': 'Refresh + check content quality — low engagement',
        'expand_and_refresh': 'Expand thin content then refresh — visible but thin',
        'monitor': 'No action now, track next cycle — model scored low',
        'protect': 'Guard a growing asset — growing page with strong visibility',
    },
    'no_go': [
        'Auto-publishing refreshes',
        'Auto-deleting pages',
        'Auto-redirecting or merging',
        'Using model probability as a KPI',
        'Applying to clients not in training data',
        'Acting on pages with < 100 impressions',
    ],
    'monitoring_triggers': {
        'precision_at_50_below': 0.60,
        'base_rate_shift_pp': 10,
        'retrain_cadence_days': 90,
    },
}

with open(WORK_OUTPUTS / 'action_summary.json', 'w') as f:
    json.dump(action_summary, f, indent=2)
print(f'Exported: action_summary.json')
print(json.dumps(action_summary, indent=2)[:1200])

In [ ]:
# --- Export: figures to work/figures/ (SVG, committed) ---
def svg_bar_chart(title, labels, values, filepath, width=900, height=450, color='#2A5D63'):
    max_val = max(values) if values else 1
    margin_l, margin_r, margin_t, margin_b = 200, 60, 60, 40
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    n = len(values)
    bar_h = max(14, (plot_h - 10 * max(n-1, 0)) / max(n, 1))
    lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    lines.append(f'<rect width="100%" height="100%" fill="#ffffff"/>')
    lines.append(f'<text x="{width/2}" y="32" text-anchor="middle" font-family="Arial" font-size="18" font-weight="600" fill="#16191C">{title}</text>')
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_t + i * (bar_h + 10)
        bar_w = (value / max_val) * plot_w
        lines.append(f'<text x="{margin_l - 12}" y="{y + bar_h*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="12" fill="#27343b">{str(label)[:35]}</text>')
        lines.append(f'<rect x="{margin_l}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{margin_l + bar_w + 8:.1f}" y="{y + bar_h*0.65:.1f}" font-family="Arial" font-size="12" fill="#27343b">{value:,.0f}</text>')
    lines.append('</svg>')
    filepath.write_text(chr(10).join(lines))
    print(f'Exported: {filepath}')

action_counts = queue['suggested_action'].value_counts()
svg_bar_chart('Suggested Action Distribution', action_counts.index.tolist(), action_counts.values.tolist(),
              WORK_FIGURES / 'action_distribution.svg')

conf_counts = queue['confidence'].value_counts().reindex(['high', 'medium', 'low'], fill_value=0)
svg_bar_chart('Confidence Distribution', conf_counts.index.tolist(), conf_counts.values.tolist(),
              WORK_FIGURES / 'confidence_distribution.svg', color='#6F4E7C')

svg_bar_chart('Top Reason Codes', reason_counts.index.tolist()[:10], reason_counts.values.tolist()[:10],
              WORK_FIGURES / 'top_reason_codes.svg', color='#B4552F')

imp_df = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False).head(10)
svg_bar_chart('Top 10 Feature Importances (Gradient Boosting)',
              imp_df['feature'].tolist(), imp_df['importance'].tolist(),
              WORK_FIGURES / 'feature_importance.svg', color='#4E79A7')

score_by_action = queue.groupby('suggested_action')['final_refresh_score'].mean().sort_values(ascending=False)
svg_bar_chart('Mean Refresh Score by Action',
              score_by_action.index.tolist(), score_by_action.values.tolist(),
              WORK_FIGURES / 'score_by_action.svg', color='#2A5D63')

In [ ]:
# --- Export: metrics receipt JSON ---
metrics_receipt = {
    'model': 'gradient_boosting',
    'random_state': RANDOM_STATE,
    'total_rows': int(len(df)),
    'feature_count': int(len(feature_cols)),
    'base_rate': float(df['is_declining_label'].mean()),
    'in_sample_roc_auc': float(roc_auc_score(y, probabilities)),
    'in_sample_average_precision': float(average_precision_score(y, probabilities)),
    'top_50_declining_rate': float(export_queue.head(50)['is_declining_label'].mean()),
    'top_100_declining_rate': float(export_queue.head(100)['is_declining_label'].mean()),
    'top_10_features': [{'feature': str(r['feature']), 'importance': float(r['importance'])} for _, r in imp_df.iterrows()],
    'action_counts': {k: int(v) for k, v in action_counts.items()},
    'note': 'In-sample metrics. See w06_validation_audit.ipynb for out-of-fold validation.',
}

with open(WORK_OUTPUTS / 'model_metrics_receipt.json', 'w') as f:
    json.dump(metrics_receipt, f, indent=2)
print(f'Exported: model_metrics_receipt.json')
print()
print('ALL EXPORTS COMPLETE:')
print(f'  work/outputs/refresh_queue.csv ({len(export_queue):,} rows)')
print(f'  work/outputs/refresh_queue_top50.csv')
print(f'  work/outputs/refresh_queue_top100.csv')
print(f'  work/outputs/action_summary.json')
print(f'  work/outputs/model_metrics_receipt.json')
print(f'  work/figures/action_distribution.svg')
print(f'  work/figures/confidence_distribution.svg')
print(f'  work/figures/top_reason_codes.svg')
print(f'  work/figures/feature_importance.svg')
print(f'  work/figures/score_by_action.svg')

## 6. Self-check

- [x] **Ranked actions with reason codes** — 6 actions, 10 reason codes, archetype mapping
- [x] **Intended use and limits** — who uses it, what they do, where it stops, cost/value
- [x] **Human review + no-go list** — 5 human-review checks, 7 no-go cases
- [x] **Monitoring / retrain triggers** — 6 drift signals, retrain cadence, monitoring metrics
- [x] **Exports for the paper** — queue CSV, top50/100, 2 JSONs, 5 SVG figures
- [x] **All claims use safe language** — observed, measured, directional, decision-support
- [x] **No client names, URLs, or private data** — all anonymized
- [x] **Committed to repo** under `work/notebooks/w07_action_playbook.ipynb`